# Inference Time Study

In [1]:
################################################## Initialize ##################################################

# Add the new path
import sys
new_path = "/home/michele/code/michele_mmdet3d/"
if not new_path in sys.path:
    sys.path.insert(1, new_path)



############################################# Check CUDA is running #############################################
import torch
if not torch.cuda.is_available():
    raise MemoryError("\t\tCUDA not available: exiting...\n\n\n")

In [2]:
############################################### Generic Parameters ###############################################

# Boolean to print inputs/outputs of each stage
wanna_print_in_out = False

# Home directory within ADE
home_dir = '/home/michele/code/'

# Relative path from "home_dir" to the pointcloud and image files
path_from_home_to_pointcloud_files = "michele_mmdet3d/data/minerva_polimove/training/velodyne_reduced"
path_from_home_to_image_files = "michele_mmdet3d/data/minerva_polimove/training/image_2"
# Path to the ".txt" file in "ImageSets", containing the list of validation files
val_list_txt_file = "/home/michele/code/michele_mmdet3d/data/minerva_polimove/ImageSets/val.txt"
# Path to the .pkl file for the validation dataset (saved during create_data.py)
pkl_info_file = "/home/michele/code/michele_mmdet3d/data/minerva_polimove/minerva_polimove_infos_val.pkl"

# Boolean to choose the type of model ("TensorRT" or "PyTorch")
# model_type = "Tensor-RT"
model_type = "PyTorch"



############################################# Configuration Parameters #############################################



# Case 0: MVX-NET SMALL VOXEL | STD CONVOLUTION
mmdet3d_cfg = home_dir + 'michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_StdConvolution/MINERVA_mvxnet_MVX_SmallVoxels_StdConvolution.py'
# Specific for Tensor-RT
if model_type == "Tensor-RT":
    deploy_cfg = home_dir + ''
    engine_files = [ home_dir + '' ]
    reference_img = home_dir + ''
# Specific for PyTorch
weights = home_dir + 'michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_StdConvolution/epoch_180_MVX_SmallVoxels_StdConvolution.pth'



# # Case 1: MVX-NET SMALL VOXEL | DEEP CONVOLUTION
# mmdet3d_cfg = home_dir + 'michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_DeepConvolution/MINERVA_mvxnet_MVX_SmallVoxels_DeepConvolution.py'
# # Specific for Tensor-RT
# if model_type == "Tensor-RT":
#     deploy_cfg = home_dir + ''
#     engine_files = [ home_dir + '' ]
#     reference_img = home_dir + ''
# # Specific for PyTorch
# weights = home_dir + 'michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_DeepConvolution/epoch_250_MVX_SmallVoxels_DeepConvolution.pth'

In [ ]:
################################################ Create the model ################################################
from mmdeploy.apis.inference import get_model
from mmdet3d.apis.inferencers import MultiModalityDet3DInferencer



if model_type == "Tensor-RT":
    # Build the model with the built-in function of MMDeploy
    model = get_model(
        model_cfg= mmdet3d_cfg,
        deploy_cfg= deploy_cfg,
        backend_files= engine_files,
        img= reference_img,
        device='cuda')
    model_cfg = model.model_cfg
    # Set the "save_losses_on_file" to False
    model.save_losses_on_file = False

elif model_type == "PyTorch":
    # Get the model from the inferencer
    inferencer = MultiModalityDet3DInferencer(model= mmdet3d_cfg,
                                              weights= weights,
                                              device= 'cuda',
                                              show_progress=False)
    model_cfg = inferencer.cfg
    model = inferencer.model
    # Set the "save_losses_on_file" to False
    model.save_losses_on_file = False



# NOTE: Build the additional BBoxHead
#
#   - Need to do it with the inferencer, otherwise some wrong weights are loaded!
#   - To access the weights, need to write "print(pts_bbox_head.conv_cls.weight)"
#
#   - If do "MODELS.build(model_cfg.model)" then the wrong weights are loaded
#
#   - Tried to check for the LiDAR-only case, but for some reason this problem is
#     not present
#
pts_bbox_head = MultiModalityDet3DInferencer(model= mmdet3d_cfg,
                                            weights= weights,
                                            device= 'cuda',
                                            show_progress=False)._init_model(model_cfg,
                                                                             weights).pts_bbox_head.to('cuda:0')

In [4]:
############################################### Automatic configs ###############################################
config_dictionary = {}

# For the loader LoadPointsFromFile
config_dictionary['LoadPointsFromFile'] = {
    'coord_type': model_cfg.val_dataloader.dataset.pipeline[0].coord_type,
    'load_dim': model_cfg.val_dataloader.dataset.pipeline[0].load_dim,
    'use_dim': model_cfg.val_dataloader.dataset.pipeline[0].use_dim
}

config_dictionary['Det3DDataPreprocessor'] = {
    'voxel': model_cfg.model.data_preprocessor.voxel,
    'voxel_type': model_cfg.model.data_preprocessor.voxel_type,
    'voxel_layer': model_cfg.model.data_preprocessor.voxel_layer,
    'mean': model_cfg.model.data_preprocessor.mean,
    'std': model_cfg.model.data_preprocessor.std,
    'bgr_to_rgb': model_cfg.model.data_preprocessor.bgr_to_rgb,
    'pad_size_divisor': model_cfg.model.data_preprocessor.pad_size_divisor
}

In [5]:
############################################### Very STARTING input ###############################################

# Read the names of the validation files from the ".txt" file in "ImageSets"
with open(val_list_txt_file, 'r') as file:
    val_file_names = sorted([line.strip() for line in file])



# Create the inputs
#   - See the details in "inference_and_model_FUSION.ipynb"
#   - For each input path to Pointcloud-Image-InfoFile
import os
inputs = []
for file_name in val_file_names:
    inputs.append([
        os.path.join(home_dir, path_from_home_to_pointcloud_files, file_name+".bin"),
        os.path.join(home_dir, path_from_home_to_image_files, file_name+".png")
    ])



# Print the output of this stage
if wanna_print_in_out:
    print("\nStarting values:")
    for element in inputs:
        print(element)

In [6]:
###################################################################################################################
###############################################  LoadPointsFromFile ###############################################
###############################################                     ###############################################
###############################################      ...and...      ###############################################
###############################################                     ###############################################
###############################################  LoadImageFromFile  ###############################################
###################################################################################################################

from mmdet3d.datasets.transforms.loading import LoadPointsFromFile
from mmcv.transforms.loading import LoadImageFromFile

# Initialize the loaders
loader_pointcloud = LoadPointsFromFile(
    coord_type=config_dictionary['LoadPointsFromFile']['coord_type'],
    load_dim=config_dictionary['LoadPointsFromFile']['load_dim'],
    use_dim=config_dictionary['LoadPointsFromFile']['use_dim']
)
loader_image = LoadImageFromFile()

# Print the input to this stage
if wanna_print_in_out:
    print("\nLoadPointsFromFile and LoadImageFromFile input:")
    for element in inputs:
        print(element)



# For cycle to also handle lists of inputs
for i in range(len(inputs)):
    
    # Prepare the string input for the loaders
    #   - The Pointcloud Loader needs two dictionaries, one nested into the other
    #   - The Image Loader needs just one dictionary
    inputs[i] = dict(
        lidar_points=dict(
            lidar_path=inputs[i][0]
        ),
        img_path=inputs[i][1]
    )

    # Actual modification of the dictionary
    loader_pointcloud(inputs[i])
    loader_image(inputs[i])



# Print the output of this stage
if wanna_print_in_out: 
    print("\nLoadPointsFromFile and LoadImageFromFile output:")
    for element in inputs:
        print(element)

In [7]:
############################################### Add METAINFO ###############################################

import mmengine
import numpy as np
import os.path as osp
from mmdet3d.structures.bbox_3d import Box3DMode, LiDARInstance3DBoxes

# Print the input of this stage
if wanna_print_in_out: 
    print("\nBefore adding metainfos:")
    for element in inputs:
        print(element)

# Get the METAINFO from the .pkl file (saved during create_data.py)
info_list = mmengine.load(pkl_info_file)['data_list']



# Check the format of info_list, and make it match inputs
new_info_list = []
if not len(info_list) == len(inputs):
    # Copied from MultiModalityDet3DInferencer._inputs_to_list()
    for element in inputs:
        timestamp = element['img_path'].split('/')[-1].split('.')[0]
        for element in info_list:
            if str(element['sample_idx']) == str(timestamp):
                new_info_list.append(element)
    # Update the field "info_list"    
    info_list = new_info_list



# Get the right fields
cam_type = "CAM2"
for index, input in enumerate(inputs):
    data_info = info_list[index]
    img_path = data_info['images'][cam_type]['img_path']
    if isinstance(input['img'], str) and \
            osp.basename(img_path) != osp.basename(input['img']):
        raise ValueError(
            f'the info file of {img_path} is not provided.')
    cam2img = np.asarray(
        data_info['images'][cam_type]['cam2img'], dtype=np.float32)
    lidar2cam = np.asarray(
        data_info['images'][cam_type]['lidar2cam'],
        dtype=np.float32)
    if 'lidar2img' in data_info['images'][cam_type]:
        lidar2img = np.asarray(
            data_info['images'][cam_type]['lidar2img'],
            dtype=np.float32)
    else:
        lidar2img = cam2img @ lidar2cam
    input['cam2img'] = cam2img
    input['lidar2cam'] = lidar2cam
    input['lidar2img'] = lidar2img
    input['box_mode_3d'] = Box3DMode.LIDAR
    input['box_type_3d'] = LiDARInstance3DBoxes

    # Added part for the addition of BBoxes
    input['gt_bboxes_3d'] = [data_info['instances'][0]['bbox_3d']]
    input['gt_labels_3d'] = data_info['instances'][0]['bbox_label']



# Print the output of this stage
if wanna_print_in_out: 
    print("\nAfter adding metainfos:")
    for element in inputs:
        print(element)

In [8]:
############################################### Pack3DDetInputs ###############################################

from mmdet3d.datasets.transforms.formating import Pack3DDetInputs

# Initialize the packer
packer = Pack3DDetInputs(keys=['points', 'img', 'gt_bboxes_3d', 'gt_labels_3d'])

# Print the input to this stage
if wanna_print_in_out: 
    print("\nPack3DDetInputs input:")
    for element in inputs:
        print(element)



# For cycle to also handle lists of inputs
for i in range(len(inputs)):
    
    # Actual modification of the dictionary
    inputs[i] = packer(inputs[i])



# Print the output of this stage
if wanna_print_in_out: 
    print("\nPack3DDetInputs output:")
    for element in inputs:
        print(element)

In [9]:
############################################### Det3DDataPreprocessor ###############################################

from mmdet3d.models.data_preprocessors.data_preprocessor import Det3DDataPreprocessor

# Initialize the preprocessor
preprocessor = Det3DDataPreprocessor(
    voxel=config_dictionary['Det3DDataPreprocessor']['voxel'],
    voxel_type=config_dictionary['Det3DDataPreprocessor']['voxel_type'],
    voxel_layer=config_dictionary['Det3DDataPreprocessor']['voxel_layer'],
    mean=config_dictionary['Det3DDataPreprocessor']['mean'],
    std=config_dictionary['Det3DDataPreprocessor']['std'],
    bgr_to_rgb=config_dictionary['Det3DDataPreprocessor']['bgr_to_rgb'],
    pad_size_divisor=config_dictionary['Det3DDataPreprocessor']['pad_size_divisor']
)

# Print the input to this stage
if wanna_print_in_out: 
    print("\nDet3DDataPreprocessor input:")
    for element in inputs:
        print(element)



# Create the list with the final inputs
final_inputs = []
for i in range(len(inputs)):    

    # Create a temporary dictionary to be passed to the preprocessor (in the right format)
    temp = {
        'data_samples': [inputs[i]['data_samples']],
        'inputs': inputs[i]['inputs']}
    temp['inputs']['points'] = [inputs[i]['inputs']['points']]
    temp['inputs']['img'] = [inputs[i]['inputs']['img']]

    # Take out the result of the Det3DDataPreprocessor
    final_inputs.append(
        preprocessor(temp))



# Move the tensors to the right device
for element in final_inputs:
    # Standard ones
    element['inputs']['points'][0] = element['inputs']['points'][0].to('cuda:0')
    element['inputs']['voxels']['voxels'] = element['inputs']['voxels']['voxels'].to('cuda:0')
    element['inputs']['voxels']['coors'] = element['inputs']['voxels']['coors'].to('cuda:0')
    element['inputs']['imgs'] = element['inputs']['imgs'].to('cuda:0')

    # Added ones for losses
    if len(element['data_samples']) == 1:
        element['data_samples'][0].gt_instances_3d.bboxes_3d = element['data_samples'][0].gt_instances_3d.bboxes_3d.to('cuda:0')
        element['data_samples'][0].gt_instances_3d.labels_3d = element['data_samples'][0].gt_instances_3d.labels_3d.to('cuda:0')
    else:
        print("\t\tError! New UNFORESEEN situation just found...\n\n\n")
        exit()

# Print the output of this stage
if wanna_print_in_out: 
    print("\nDet3DDataPreprocessor output:")
    for element in final_inputs:
        print(element)

In [ ]:
################################################# Compute losses #################################################

losses=[]
for element in final_inputs:

    # Unpacking of the important elements
    batch_input_metas = [element['data_samples'][0].metainfo]
    voxel_dict = element['inputs'].get('voxels', None)
    imgs = element['inputs'].get('imgs', None)
    points = element['inputs'].get('points', None)

    # Image features extraction
    img_feats = model.extract_img_feat(imgs, batch_input_metas)

    # Points --> DynamicVFE and PointFusion
    voxel_features, feature_coors = model.pts_voxel_encoder(voxel_dict['voxels'], voxel_dict['coors'], points, img_feats, batch_input_metas)

    # Points --> SparseEncoder, SECOND, SECONDFPN
    batch_size = voxel_dict['coors'][-1, 0] + 1
    x = model.pts_middle_encoder(voxel_features, feature_coors, batch_size)
    x = model.pts_backbone(x)
    pts_feats = model.pts_neck(x)

    # Compute the stuff necessary for the loss computation
    cls_score, bbox_pred, dir_cls_pred = pts_bbox_head.forward(pts_feats)

    # Clean variables to avoid OutOfMemoryError
    img_feats = voxel_features = feature_coors = x = pts_feats = None

    # Compute the actual losses
    loss_dict = pts_bbox_head.loss_by_feat(cls_score,
                                           bbox_pred,
                                           dir_cls_pred,
                                           batch_gt_instances_3d = [element['data_samples'][0].gt_instances_3d],
                                           batch_input_metas = [element['data_samples'][0].metainfo])

    # # NOTE: Different option to compute losses 
    # #   --> "Standard" way with the inferencer
    # #   --> Does NOT work with Tensor-RT
    # loss_dict = model.loss(element['inputs'], element['data_samples'])

    # Re-arrange the losses into a dictionary
    losses.append({
        'loss_cls': float(loss_dict['loss_cls'][0]),
        'loss_bbox': float(loss_dict['loss_bbox'][0]),
        'loss_dir': float(loss_dict['loss_dir'][0]),
        'total_loss': float(loss_dict['loss_cls'][0]) + float(loss_dict['loss_bbox'][0]) + float(loss_dict['loss_dir'][0])
    })

    # Clean variables to avoid OutOfMemoryError
    cls_score = bbox_pred = dir_cls_pred = loss_dict = None

In [11]:
#################################################################################################################
#                                              PREPARE THE LISTS                                                #
#################################################################################################################

from plotters import *

# Create the lists
losses_cls = []
losses_bbox = []
losses_dir = []
losses_total = []

# Assign the values
for element in losses:
    losses_cls.append(element['loss_cls'])
    losses_bbox.append(element['loss_bbox'])
    losses_dir.append(element['loss_dir'])
    losses_total.append(element['total_loss'])

# Check that the vectors are all of the same dimension
if len(losses_cls) != len(losses_bbox) or len(losses_cls) != len(losses_dir) or len(losses_cls) != len(losses_total):
    print("\nWrong dimensions for lists!!!\n")
    exit()

In [ ]:
# Plot the total loss

freq_plot_with_variance(losses_total, "Total loss", "blue", 30)

In [ ]:
# Plot the classification loss

freq_plot_with_variance(losses_cls, "Classification loss", "green", 30)

In [ ]:
#Plot the bbox loss

freq_plot_with_variance(losses_bbox, "B-Box loss", "gold", 30)

In [ ]:
#Plot the direction loss

freq_plot_with_variance(losses_dir, "Direction loss", "black", 30)

In [ ]:
# Plot the pie chart with the percentages

plot_pie_chart(losses_cls, losses_bbox, losses_dir, "Classification", "B-Box", "Direction")